# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saleha65/Machine-Learning-by-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Paste your HF read token: ")

Paste your HF read token: ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from datasets import load_dataset
import pandas as pd

# Sirf March 2026 ki files download karega — poora 78.8M rows nahi
ds_fact = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files="**/month=2026-03/*.parquet"
)
fact_df = ds_fact["train"].to_pandas()

ds_dim = load_dataset("FlyRank/internship-warehouse", "dim_content")
dim_df = ds_dim["train"].to_pandas()

fact_df["report_date"] = pd.to_datetime(fact_df["report_date"])
month_df = fact_df.copy()

print(month_df.shape)
month_df.head()

Using the latest cached version of the dataset since FlyRank/internship-warehouse couldn't be found on the Hugging Face Hub


ValueError: Couldn't find cache for FlyRank/internship-warehouse for config 'fact_content_daily_performance-caf4b95dcdc5a833'
Available configs in the cache: ['dim_content', 'fact_content_daily_performance']

Unit of analysis: one row = one content page (content_id) for one client, on one report_date.
Time window: month = 2026-03 (mid-panel month, filtered from report_date).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
fields = {
    "feature": ["days_since_last_update", "impressions", "clicks", "avg_position", "sessions"],
    "label": ["refresh_priority_score"],
    "context": ["content_hash_id", "client_hash_id", "report_date"],
    "excluded": ["pages_with_zero_impressions_whole_month"]
}
fields

Feature: days_since_last_update, impressions, clicks, avg_position, sessions
Label/proxy: refresh_priority_score — built from staleness + declining avg_position + declining sessions
Context: content_hash_id, client_hash_id, report_date
Excluded: pages with zero impressions across the whole month — no signal to rank on

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# 1. Grain check — one row really is one page per client per day
dupes = month_df.groupby(["content_hash_id", "client_hash_id", "report_date"]).size()
print("Grain violations:", (dupes > 1).sum())

# 2. Row count + date span
print("Total rows:", len(month_df))
print("Date span:", month_df["report_date"].min(), "to", month_df["report_date"].max())

# 3. Availability — NULL vs 0 alag hain
available = month_df[month_df["impressions"].notna()]
print(f"{len(available)} of {len(month_df)} rows have non-NULL impressions")

Grain check: [X] violations found — confirms one row = one page per client per day (or flags where it doesn't).
Row count: [X] rows spanning [start date] to [end date].
Availability: [X] of [Y] rows have real (non-NULL) impressions data — the rest were never recorded, not zero.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell you: whether a refresh actually caused better rankings (correlational, not causal); performance for clients or pages outside this warehouse; or true behavior in months with GSC-only early data or overlapping windows.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.